In [36]:
# 🔧 CRM Hygiene Automation — End-to-End Pipeline
#
# **Flow:** Darwin Query → Python Rules → Outputs → Power Automate → Manager Emails

In [61]:
import ast
import json
import shutil
from pathlib import Path
from datetime import datetime, date

import pandas as pd
from linkedin.mail.liemail import LiEmail

# Format float outputs in pandas
pd.options.display.float_format = "{:.2f}".format

# ============================================================
# CONFIGURATION — update these paths for your environment
# ============================================================

RULES_VERSION = "v0.1"
MONEY_DECIMALS = 2
STAGNANT_STAGE_DAYS_THRESHOLD = 90

# TEST MODE CONFIGURATION
# If True, all manager emails in the JSON payload will be replaced by TEST_EMAIL.
# This prevents accidentally spamming managers while testing the Power Automate flow.
TEST_MODE = True
TEST_EMAIL = "gbuehrer@linkedin.com"

# Run metadata
RUN_DATE = date.today()
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
FILE_DATE = datetime.now().strftime("%m%d%y")

BASE_PATH = Path("/home/jovyan/gbuehrer")

# Output paths (local notebook workspace)
OUTPUT_DIR = BASE_PATH / "output_auto"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMAIL_TO = ["gbuehrer@linkedin.com"]
EMAIL_SUBJECT = "CRM Hygiene Base Data"

print(f"   Configuration loaded")
print(f"   Rules version:  {RULES_VERSION}")
print(f"   Test Mode:      {'ON (Emails redirected)' if TEST_MODE else 'OFF (Production)'}")
print(f"   Run date:       {RUN_DATE}")
print(f"   Run ID:         {RUN_ID}")
print(f"   Output dir:     {OUTPUT_DIR}")

In [62]:
# ============================================================
# SCHEMA DEFINITIONS
# ============================================================

REQUIRED_COLUMNS = [
    "opp_id", "final_owner_name", "opp_name", "account_name",
    "stage", "stage_days", "close_Year_F", "close_quarter_F",
    "close_date", "Forecast", "renewal_target_amt_conv",
    "WorstCase_Annualized_usd", "Upside_Annualized_usd",
]

RECOMMENDED_COLUMNS = [
    "owner_id", "owner_employee_id", "Owner_Manager",
    "ult_parent_account_name", "ult_parent_account_id",
    "sfdc_account_id", "opportunity_status", "final_rn",
    "is_temporary_coverage", "merge_source",
]

OPTIONAL_DEFAULTS = {
    "owner_id": "", "owner_employee_id": "", "Owner_Manager": "",
    "ult_parent_account_name": "", "ult_parent_account_id": "",
    "sfdc_account_id": "", "opportunity_status": "", "final_rn": "",
    "is_temporary_coverage": "", "merge_source": "",
}

MONEY_COLUMNS = [
    "Forecast", "renewal_target_amt_conv",
    "WorstCase_Annualized_usd", "Upside_Annualized_usd",
]

RULE_COLUMNS = [
    "rule_2_ol_lt_rta", "rule_3_ol_lt_downside",
    "rule_4_upside_lt_ol", "rule_5_close_date_past",
    "rule_6_missing_ol_and_us", "rule_7_stagnant_opportunity",
]

RULE_LABELS = {
    "rule_2_ol_lt_rta": "Outlook < Renewal Target Amount",
    "rule_3_ol_lt_downside": "Outlook < Downside",
    "rule_4_upside_lt_ol": "Upside < Outlook",
    "rule_5_close_date_past": "Close Date < Today",
    "rule_6_missing_ol_and_us": "Opp has no Outlook and Upside",
    "rule_7_stagnant_opportunity": "Opp in same sales stage for 90+ days",
}

print(f" {len(RULE_COLUMNS)} rules defined")

In [63]:
%reload_ext linkedin.lisql
%config SqlMagic.autocommit=False
%manage_trino holdem

In [64]:
%sql SET SESSION li_authorization_user = 'lssops'

In [65]:
%%sql base_data <<

-- ==============================================================================
-- 1. FISCAL CONTEXT & TARGETS
-- Calculates current fiscal year/quarter and defines the dynamic FY string 
-- and the list of target managers for the automation.
-- ==============================================================================
WITH fiscal_context AS (
    SELECT
        CASE
            WHEN MONTH(CURRENT_DATE) >= 7 THEN YEAR(CURRENT_DATE) + 1
            ELSE YEAR(CURRENT_DATE)
        END AS current_fiscal_year,
        CASE
            WHEN MONTH(CURRENT_DATE) BETWEEN 7 AND 9 THEN 1
            WHEN MONTH(CURRENT_DATE) BETWEEN 10 AND 12 THEN 2
            WHEN MONTH(CURRENT_DATE) BETWEEN 1 AND 3 THEN 3
            WHEN MONTH(CURRENT_DATE) BETWEEN 4 AND 6 THEN 4
        END AS current_fiscal_quarter
),

target_config AS (
    SELECT 
        -- Creates dynamic string like 'FY26 LSS GAM Account' based on fiscal year
        'FY' || CAST(current_fiscal_year % 100 AS VARCHAR) || ' LSS GAM Account' AS dynamic_fy_account_name
    FROM fiscal_context
),

target_managers AS (
    -- Centralized list of managers allowed in this automation
    SELECT fullname FROM (VALUES 
        ('Katherine Brinkman'), ('Hui Yen Ko'), ('Thibaud Savouré'), 
        ('Erin Mathurin'), ('Randy Petway')
    ) AS t(fullname)
),

-- ==============================================================================
-- 2. OPP DATA (EARLY FILTERING)
-- Extracts ONLY 'Open' opportunities. Financial metrics are simplified 
-- because we no longer need to handle 'Closed Won' logic here.
-- ==============================================================================
opp_data AS (
    SELECT
        opp.opportunity_id,
        opp.sfdc_account_id,
        opp.sales_org,
        opp.opportunity_type,
        opp.opportunity_sub_type,
        opp.opportunity_stage_name,
        opp.stage_days,
        opp.status AS opportunity_status,
        opp.fiscal_year_closed,
        opp.fiscal_quarter_closed,
        opp.first_year_amount_usd,
        opp.owner_name,
        opp.ownerid,
        opp.employeenumber AS owner_employee_id,
        opp.account_name,
        opp.close_date,
        opp.probability,
        opp.opportunity_name,
        opp.upside_annualized_usd,
        opp.worstcase_annualized_usd,
        opp.nextstep,
        opp.opportunity_created_date,
        opp.close_timestamp,
        opp.customer_urn,
        opp.owner_current_manager_id,
        opp.lss_named_account,
        
        -- Simplified metrics for Open Opportunities
        COALESCE(opp.total_outlook_amount_usd, 0) AS Open_Outlook_Amt,
        COALESCE(opp.total_outlook_first_year_amount_usd, 0) AS Forecast,
        COALESCE(opp.renewal_target_amount_usd, 0) AS renewal_target_amt_conv,
        COALESCE(opp.baseline_cy_amount_usd, 0) AS Baseline_Cy_con

    FROM u_lssops.opportunity opp
    WHERE opp.status = 'Open' -- Improvement: Filtered early for performance
),

-- ==============================================================================
-- 3. JOINED ALL (OPTIMIZED JOINS)
-- Enriches data using INNER JOIN with target managers to discard 
-- out-of-scope data immediately.
-- ==============================================================================
joined_all AS (
    SELECT
        opp.*,
        acc.ultimate_parent_name,
        acc.ultimate_parent_id,
        mgr.fullname AS Owner_Manager,
        mgr.email AS manager_email

    FROM opp_data opp
    INNER JOIN u_lssops.account acc ON acc.customer_urn = opp.customer_urn
    -- Improvement: Inner Join with the specific target list
    INNER JOIN u_salesbi.users mgr ON mgr.sfdc_user_id = opp.owner_current_manager_id
    INNER JOIN target_managers tm ON mgr.fullname = tm.fullname
    CROSS JOIN target_config tc

    WHERE opp.lss_named_account = tc.dynamic_fy_account_name -- Improvement: Dynamic FY
      AND opp.sales_org = 'GAM'
),

-- ==============================================================================
-- 4. DEDUPLICATION
-- Keep only the primary record per opportunity.
-- ==============================================================================
final_ranked AS (
    SELECT j.*,
        ROW_NUMBER() OVER (
            PARTITION BY j.opportunity_id
            ORDER BY j.close_timestamp DESC
        ) AS final_rn
    FROM joined_all j
),

deduped AS (
    SELECT * FROM final_ranked WHERE final_rn = 1
),

-- ==============================================================================
-- 5. FINAL BASE & OUTPUT
-- Applies fiscal window filtering (Next 3 Quarters) and final column formatting.
-- ==============================================================================
final_base AS (
    SELECT d.*,
        (d.fiscal_year_closed * 4 + CAST(REGEXP_EXTRACT(d.fiscal_quarter_closed, 'Q([1-4])', 1) AS INTEGER)) AS close_fiscal_period_index,
        fc.current_fiscal_year,
        fc.current_fiscal_quarter,
        (fc.current_fiscal_year * 4 + fc.current_fiscal_quarter) AS current_fiscal_period_index
    FROM deduped d
    CROSS JOIN fiscal_context fc
)

SELECT
    opportunity_id AS opp_id,
    ownerid AS owner_id,
    owner_employee_id,
    opportunity_status,
    owner_name AS final_owner_name,
    opportunity_name AS opp_name,
    ultimate_parent_name AS ult_parent_account_name,
    ultimate_parent_id AS ult_parent_account_id,
    account_name,
    sfdc_account_id,
    Owner_Manager,
    manager_email,
    opportunity_stage_name AS stage,
    stage_days,
    fiscal_year_closed AS close_Year_F,
    fiscal_quarter_closed AS close_quarter_F,
    close_date,
    Forecast,
    renewal_target_amt_conv,
    Baseline_Cy_con,
    probability / 100 AS Probability,
    COALESCE(upside_annualized_usd, 0) AS Upside_Annualized_usd,
    COALESCE(worstcase_annualized_usd, 0) AS WorstCase_Annualized_usd,
    nextstep,
    DATE_DIFF('day', opportunity_created_date, CURRENT_DATE) AS days_open

FROM final_base

WHERE close_fiscal_period_index BETWEEN current_fiscal_period_index AND current_fiscal_period_index + 3
ORDER BY opp_id

In [66]:
# ============================================================
# QUERY OUTPUT CONVERSION
# ============================================================

def convert_query_output_to_df(query_output):
    """Converts Darwin / SQL magic output into a pandas DataFrame."""
    if isinstance(query_output, pd.DataFrame):
        return query_output.copy()
    if hasattr(query_output, "DataFrame"):
        return query_output.DataFrame()
    if hasattr(query_output, "to_pandas"):
        return query_output.to_pandas()
    return pd.DataFrame(query_output)

df = convert_query_output_to_df(base_data)
print(f" Query returned {len(df)} rows, {len(df.columns)} columns")

In [67]:
# ============================================================
# SCHEMA VALIDATION & DATA PREP
# ============================================================

def normalize_columns(df):
    """Strips whitespace from column names to prevent key errors."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def validate_schema(df):
    """Ensures all required columns are present in the SQL output."""
    missing_required = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    missing_recommended = [c for c in RECOMMENDED_COLUMNS if c not in df.columns]
    if missing_required:
        raise ValueError(f"Missing required columns: {missing_required}")
    return {
        "missing_required_columns": missing_required,
        "missing_recommended_columns": missing_recommended,
    }

def ensure_optional_columns(df):
    """Adds missing optional columns with default values."""
    df = df.copy()
    for col, default in OPTIONAL_DEFAULTS.items():
        if col not in df.columns:
            df[col] = default
    return df

def enforce_types(df):
    """Cleans up data types and formats money columns."""
    df = df.copy()
    df["opp_id"] = df["opp_id"].astype(str).str.strip()
    df["final_owner_name"] = df["final_owner_name"].astype(str).str.strip()
    df["owner_id"] = df["owner_id"].astype(str).str.strip()
    df["owner_employee_id"] = df["owner_employee_id"].astype(str).str.strip()
    df["Owner_Manager"] = df["Owner_Manager"].astype(str).str.strip()
    df["stage"] = df["stage"].astype(str).str.strip()
    df["close_date"] = pd.to_datetime(df["close_date"], errors="coerce").dt.date
    for col in MONEY_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).round(MONEY_DECIMALS)
    return df

def validate_base_data_quality(df):
    """Generates a summary of data quality for the execution log."""
    return {
        "total_rows": int(len(df)),
        "duplicate_opp_id_rows": int(df["opp_id"].duplicated().sum()),
        "null_counts": {col: int(df[col].isna().sum()) for col in REQUIRED_COLUMNS if col in df.columns},
        "final_rn_distribution": ({str(k): int(v) for k, v in df["final_rn"].value_counts(dropna=False).items()} if "final_rn" in df.columns else None),
        "opportunity_status_distribution": ({str(k): int(v) for k, v in df["opportunity_status"].value_counts(dropna=False).items()} if "opportunity_status" in df.columns else None),
        "fiscal_distribution": (df.groupby(["close_Year_F", "close_quarter_F"], dropna=False).size().reset_index(name="opp_count").to_dict("records")),
    }


# ============================================================
# HYGIENE RULES (2-6)
# ============================================================

def apply_rules_2_to_6(df, run_date):
    """Applies the core logical validations for opportunity hygiene."""
    df = df.copy()
    
    # Financial Rules
    df["rule_2_ol_lt_rta"] = df["Forecast"] < df["renewal_target_amt_conv"]
    df["rule_3_ol_lt_downside"] = df["Forecast"] < df["WorstCase_Annualized_usd"]
    df["rule_4_upside_lt_ol"] = df["Upside_Annualized_usd"] < df["Forecast"]
    
    # IMPROVEMENT: Vectorized date comparison for better performance
    df["rule_5_close_date_past"] = pd.to_datetime(df["close_date"], errors="coerce").dt.date < run_date
    
    # Missing Value Rules
    df["missing_ol"] = df["Forecast"] == 0
    df["missing_us"] = df["Upside_Annualized_usd"] == 0
    df["rule_6_missing_ol_and_us"] = df["missing_ol"] & df["missing_us"]

    def get_rule_6_detail(row):
        if row["missing_ol"] and row["missing_us"]:
            return "Missing OL and US"
        return ""

    df["rule_6_detail"] = df.apply(get_rule_6_detail, axis=1)
    return df

# ============================================================
# HYGIENE RULE 7 — Stagnant Opportunity (stage_days)
# ============================================================

def parse_stage_days(stage_days_value):
    """Parses the stage_days dictionary returned from the SQL query."""
    if pd.isna(stage_days_value):
        return {}
    if isinstance(stage_days_value, dict):
        return stage_days_value
    try:
        # Attempts to parse string representations of dicts safely
        parsed = ast.literal_eval(str(stage_days_value))
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}

def get_days_in_current_stage(stage_days_value, current_stage):
    """Extracts the number of days the opportunity has been in its current stage."""
    if pd.isna(current_stage):
        return None
    stage_dict = parse_stage_days(stage_days_value)
    if not isinstance(stage_dict, dict):
        return None
    if current_stage in stage_dict:
        return stage_dict.get(current_stage)
    
    current_stage_normalized = str(current_stage).strip().lower()
    for stage_name, days in stage_dict.items():
        if str(stage_name).strip().lower() == current_stage_normalized:
            return days
    return None

def apply_rule_7_stagnant_opportunity(df):
    """Flags opportunities that haven't moved stages in over 30 days."""
    df = df.copy()
    df["days_in_current_stage"] = df.apply(
        lambda row: get_days_in_current_stage(row.get("stage_days"), row.get("stage")),
        axis=1
    )
    df["days_in_current_stage"] = pd.to_numeric(df["days_in_current_stage"], errors="coerce")
    
    # Flag condition
    df["rule_7_stagnant_opportunity"] = (
        df["days_in_current_stage"].notna()
        & (df["days_in_current_stage"] >= STAGNANT_STAGE_DAYS_THRESHOLD)
    )
    
    rule_7_status = {
        "rule_7_logic": "current stage unchanged for 90+ days based on stage_days",
        "stage_days_threshold": STAGNANT_STAGE_DAYS_THRESHOLD,
        "opps_with_stage_days_available": int(df["days_in_current_stage"].notna().sum()),
        "rule_7_flagged_opps": int(df["rule_7_stagnant_opportunity"].sum()),
    }
    return df, rule_7_status

# ============================================================
# FLAG AGGREGATION
# ============================================================

def add_flag_columns(df):
    """Aggregates all triggered rules into summary text and boolean columns."""
    df = df.copy()
    for col in RULE_COLUMNS:
        if col not in df.columns:
            df[col] = False
        df[col] = df[col].fillna(False).astype(bool)

    def all_flags(row):
        return [RULE_LABELS[col] for col in RULE_COLUMNS if row[col]]

    df["all_flags"] = df.apply(all_flags, axis=1)
    df["all_flags_text"] = df["all_flags"].apply(lambda x: "; ".join(x))
    df["total_flags"] = df[RULE_COLUMNS].sum(axis=1).astype(int)
    df["is_flagged"] = df["total_flags"] > 0
    return df

# ============================================================
# SAFE TYPE CONVERTERS (Prevents NaN issues in JSON)
# ============================================================

def safe_float(value):
    try:
        return None if pd.isna(value) else float(value)
    except Exception:
        return None

def safe_int(value):
    try:
        return None if pd.isna(value) else int(value)
    except Exception:
        return None

def safe_str(value):
    try:
        return None if pd.isna(value) else str(value)
    except Exception:
        return None

# ============================================================
# JSON PAYLOAD BUILDER (For Power Automate)
# ============================================================

def build_manager_payload(flagged_df):
    """Constructs the structured JSON payload grouped by manager."""
    payload = []
    if flagged_df.empty:
        return payload

    MANAGER_CC_MAP = {
        "Katherine Brinkman": "rchakravarthy@linkedin.com",
        "Hui Yen Ko": "prajani@linkedin.com",
        "Thibaud Savouré": "mcastet@linkedin.com; giowang@linkedin.com",
        "Erin Mathurin": "rchakravarthy@linkedin.com",
        "Randy Petway": "mcastet@linkedin.com; giowang@linkedin.com"
    }

    for manager_name, manager_df in flagged_df.groupby("Owner_Manager", dropna=False):
        manager_name_clean = (
            "Unknown Manager"
            if pd.isna(manager_name) or str(manager_name).strip() == ""
            else str(manager_name)
        )
        
        # Retrives the actual manager email from the SQL extraction
        real_email = manager_df["manager_email"].dropna().iloc[0] if not manager_df["manager_email"].dropna().empty else None
        real_cc = MANAGER_CC_MAP.get(manager_name_clean, "")
        
        # Applies Test Mode logic: overrides email if TEST_MODE = True
        if TEST_MODE:
            manager_email = TEST_EMAIL
            manager_cc = TEST_EMAIL if real_cc != "" else "" 
        else:
            manager_email = real_email
            manager_cc = real_cc
        
        # Sort opportunities by owner, then severity, then date
        manager_df = manager_df.sort_values(
            ["final_owner_name", "total_flags", "close_date", "account_name", "opp_name"],
            ascending=[True, False, True, True, True],
        )

        opportunities = []
        for _, row in manager_df.iterrows():
            opportunities.append({
                "owner_name": safe_str(row.get("final_owner_name")),
                "owner_id": safe_str(row.get("owner_id")),
                "owner_employee_id": safe_str(row.get("owner_employee_id")),
                "opp_id": safe_str(row.get("opp_id")),
                "opp_name": safe_str(row.get("opp_name")),
                "account_name": safe_str(row.get("account_name")),
                "ult_parent_account_name": safe_str(row.get("ult_parent_account_name")),
                "stage": safe_str(row.get("stage")),
                "days_in_current_stage": safe_int(row.get("days_in_current_stage")),
                "close_date": safe_str(row.get("close_date")),
                "all_flags_text": safe_str(row.get("all_flags_text")),
                "all_flags": row.get("all_flags") if isinstance(row.get("all_flags"), list) else [],
                "total_flags": safe_int(row.get("total_flags")),
                "rule_6_detail": safe_str(row.get("rule_6_detail")),
                "forecast": safe_float(row.get("Forecast")),
                "renewal_target_amt_conv": safe_float(row.get("renewal_target_amt_conv")),
                "downside": safe_float(row.get("WorstCase_Annualized_usd")),
                "upside": safe_float(row.get("Upside_Annualized_usd")),
            })

        issue_summary = {}
        for flags_text in manager_df["all_flags_text"].dropna():
            for flag in str(flags_text).split(";"):
                flag_clean = flag.strip()
                if flag_clean:
                    issue_summary[flag_clean] = issue_summary.get(flag_clean, 0) + 1

        payload.append({
            "manager_name": manager_name_clean,
            "manager_email": manager_email,
            "manager_cc": manager_cc,
            "total_flagged_opps": int(manager_df["opp_id"].nunique()),
            "owners_impacted": int(manager_df["final_owner_name"].nunique()),
            "issue_summary": issue_summary,
            "opportunities": opportunities,
        })

    # Returns managers with the most flagged opportunities first
    return sorted(payload, key=lambda x: x["total_flagged_opps"], reverse=True)

# ============================================================
# AUDIT LOG BUILDER
# ============================================================

def build_run_log(df, flagged_df, schema_validation, data_quality, rule_7_status,
                  run_id, run_date, input_source):
    """Compiles execution metrics and statistics into a log dictionary."""
    rule_counts = {
        RULE_LABELS[col]: int(df[col].sum())
        for col in RULE_COLUMNS if col in df.columns
    }
    rule_6_detail_counts = (
        flagged_df["rule_6_detail"].value_counts(dropna=False).to_dict()
        if "rule_6_detail" in flagged_df.columns and not flagged_df.empty
        else {}
    )
    return {
        "run_id": run_id,
        "run_date": run_date.isoformat(),
        "rules_version": RULES_VERSION,
        "input_source": str(input_source),
        "schema_validation": schema_validation,
        "data_quality": data_quality,
        "total_opps_in_scope": int(len(df)),
        "distinct_flagged_opps": int(len(flagged_df)),
        "total_rule_hits": int(df[RULE_COLUMNS].sum().sum()),
        "rule_counts": rule_counts,
        "rule_6_detail_counts": {str(k): int(v) for k, v in rule_6_detail_counts.items() if str(k)},
        "owners_impacted": int(flagged_df["final_owner_name"].nunique() if not flagged_df.empty else 0),
        "managers_impacted": int(flagged_df["Owner_Manager"].nunique() if not flagged_df.empty else 0),
        "rule_7_status": rule_7_status,
    }

print(" All rule functions loaded (v0.1)")

In [68]:
# ============================================================
# PIPELINE EXECUTION
# ============================================================

source_name = f"darwin_query_{RUN_DATE.isoformat()}"

# --- Schema & types ---
df = normalize_columns(df)
schema_validation = validate_schema(df)
df = ensure_optional_columns(df)
df = enforce_types(df)
data_quality = validate_base_data_quality(df)

print(f" Base Data: {data_quality['total_rows']} rows, "
      f"{data_quality['duplicate_opp_id_rows']} duplicate opp_ids")

# --- Apply rules ---
df = apply_rules_2_to_6(df, RUN_DATE)
df, rule_7_status = apply_rule_7_stagnant_opportunity(df)
df = add_flag_columns(df)

# Filter only the flagged opportunities for the outputs
flagged_df = df[df["is_flagged"]].copy()

print()
print(f"   Flagged opps:      {len(flagged_df)} / {len(df)}")
print(f"   Total rule hits:   {int(df[RULE_COLUMNS].sum().sum())}")
print(f"   Owners impacted:   {flagged_df['final_owner_name'].nunique() if not flagged_df.empty else 0}")
print(f"   Managers impacted: {flagged_df['Owner_Manager'].nunique() if not flagged_df.empty else 0}")
print(f"   Rule 7 (stagnant): {rule_7_status['rule_7_flagged_opps']}")

# --- Build outputs ---
manager_payload = build_manager_payload(flagged_df)
run_log = build_run_log(
    df=df, flagged_df=flagged_df,
    schema_validation=schema_validation,
    data_quality=data_quality,
    rule_7_status=rule_7_status,
    run_id=RUN_ID, run_date=RUN_DATE,
    input_source=source_name,
)

print()
print(f" Manager payload: {len(manager_payload)} managers")
print(f" Pipeline complete!")

In [69]:
# ============================================================
# SAVE OUTPUTS (With dated filenames for auditing)
# ============================================================

run_date_str = RUN_DATE.isoformat()  # e.g. "2026-05-07"

# --- Flagged Opportunities ---
flagged_path = OUTPUT_DIR / f"flagged_opps_{run_date_str}.xlsx"

with pd.ExcelWriter(flagged_path, engine="openpyxl") as writer:
    flagged_df.to_excel(writer, sheet_name="flagged_opps", index=False)

    # Generate summary tabs if data exists
    if not flagged_df.empty:
        (
            flagged_df.groupby(["Owner_Manager"], dropna=False)
            .agg(
                total_flagged_opps=("opp_id", "nunique"),
                owners_impacted=("final_owner_name", "nunique"),
                total_rule_hits=("total_flags", "sum"),
            )
            .reset_index()
            .sort_values("total_flagged_opps", ascending=False)
        ).to_excel(writer, sheet_name="manager_summary", index=False)

        (
            flagged_df.groupby(
                ["final_owner_name", "owner_id", "owner_employee_id", "Owner_Manager"],
                dropna=False,
            )
            .agg(
                total_flagged_opps=("opp_id", "nunique"),
                total_rule_hits=("total_flags", "sum"),
            )
            .reset_index()
            .sort_values("total_flagged_opps", ascending=False)
        ).to_excel(writer, sheet_name="owner_summary", index=False)

    # Output rule distribution
    pd.DataFrame([
        {"rule": RULE_LABELS[col], "count": int(df[col].sum())}
        for col in RULE_COLUMNS
    ]).to_excel(writer, sheet_name="rule_summary", index=False)

    if "rule_6_detail" in flagged_df.columns and not flagged_df.empty:
        r6 = flagged_df["rule_6_detail"].value_counts(dropna=False).reset_index()
        r6.columns = ["rule_6_detail", "count"]
        r6.to_excel(writer, sheet_name="rule_6_detail_summary", index=False)

    if "days_in_current_stage" in flagged_df.columns and not flagged_df.empty:
        stagnant = flagged_df[flagged_df["rule_7_stagnant_opportunity"]]
        if not stagnant.empty:
            (
                stagnant.groupby(["stage"], dropna=False)
                .agg(
                    total_stagnant_opps=("opp_id", "nunique"),
                    avg_days=("days_in_current_stage", "mean"),
                    max_days=("days_in_current_stage", "max"),
                )
                .reset_index()
                .sort_values("total_stagnant_opps", ascending=False)
            ).to_excel(writer, sheet_name="stagnant_summary", index=False)

print(f" {flagged_path}")

# --- Manager Payload ---
payload_path = OUTPUT_DIR / f"manager_payload_{run_date_str}.json"
payload_path.write_text(
    json.dumps(manager_payload, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print(f" {payload_path}")

# --- Run Log ---
run_log_path = OUTPUT_DIR / f"run_log_{run_date_str}.json"
run_log_path.write_text(
    json.dumps(run_log, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print(f" {run_log_path}")

# --- Base Data Backup ---
base_data_path = OUTPUT_DIR / f"CRM_Hygiene_Base_Data_{run_date_str}.xlsx"
with pd.ExcelWriter(base_data_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Base Data", index=False)
print(f" {base_data_path}")

In [70]:
# ============================================================
# EMAIL TRIGGER -> POWER AUTOMATE
# ============================================================
#
# Flow:
#   1. LiEmail sends all files as attachments.
#   2. Power Automate trigger: "When a new email arrives"
#      - Filter: Subject contains "CRM Hygiene Run"
#   3. Power Automate saves attachments to OneDrive.
#   4. Power Automate parses manager_payload.json and loops through 
#      it to send the HTML tables to the corresponding managers.
# ============================================================

from linkedin.mail.liemail import LiEmail

email_body = f"""
<!DOCTYPE html>
<html>
<body style="font-family: Arial, sans-serif; color: #333;">
    <h2>🔧 CRM Hygiene Automation — Run Complete</h2>

    <table style="border-collapse: collapse; margin: 16px 0;">
        <tr><td style="padding: 4px 16px 4px 0; font-weight: bold;">Run ID</td><td>{RUN_ID}</td></tr>
        <tr><td style="padding: 4px 16px 4px 0; font-weight: bold;">Run Date</td><td>{RUN_DATE}</td></tr>
        <tr><td style="padding: 4px 16px 4px 0; font-weight: bold;">Rules Version</td><td>{RULES_VERSION}</td></tr>
        <tr><td style="padding: 4px 16px 4px 0; font-weight: bold;">Test Mode</td><td>{'Active (All emails redirected)' if TEST_MODE else 'Inactive'}</td></tr>
        <tr><td style="padding: 4px 16px 4px 0; font-weight: bold;">Opps in Scope</td><td>{run_log['total_opps_in_scope']}</td></tr>
        <tr><td style="padding: 4px 16px 4px 0; font-weight: bold;">Flagged Opps</td><td>{run_log['distinct_flagged_opps']}</td></tr>
        <tr><td style="padding: 4px 16px 4px 0; font-weight: bold;">Total Rule Hits</td><td>{run_log['total_rule_hits']}</td></tr>
        <tr><td style="padding: 4px 16px 4px 0; font-weight: bold;">Owners Impacted</td><td>{run_log['owners_impacted']}</td></tr>
        <tr><td style="padding: 4px 16px 4px 0; font-weight: bold;">Managers Impacted</td><td>{run_log['managers_impacted']}</td></tr>
        <tr><td style="padding: 4px 16px 4px 0; font-weight: bold;">Stagnant Opps</td><td>{rule_7_status['rule_7_flagged_opps']}</td></tr>
    </table>

    <h3>📎 Attachments</h3>
    <ul>
        <li><b>manager_payload_{run_date_str}.json</b> — payload for Power Automate emails</li>
        <li><b>flagged_opps_{run_date_str}.xlsx</b> — all flagged opportunities + summaries</li>
        <li><b>run_log_{run_date_str}.json</b> — full audit log</li>
        <li><b>CRM_Hygiene_Base_Data_{run_date_str}.xlsx</b> — raw Base Data backup</li>
    </ul>

    <p style="color: #888; font-size: 12px;">
        This email was sent automatically by the CRM Hygiene Automation pipeline.
        Power Automate will save these files to OneDrive and process the manager payload.
    </p>
</body>
</html>
"""

with LiEmail() as le:
    le.sender = "gbuehrer@linkedin.com"
    le.recipients = ["gbuehrer@linkedin.com"]
    le.subject = f"CRM Hygiene Run {run_date_str}"
    le.email_body = email_body
    le.content_type = "html"

    # Attach all generated output files
    le.attach_file(str(payload_path), f"manager_payload_{run_date_str}.json")
    le.attach_file(str(flagged_path), f"flagged_opps_{run_date_str}.xlsx")
    le.attach_file(str(run_log_path), f"run_log_{run_date_str}.json")
    le.attach_file(str(base_data_path), f"CRM_Hygiene_Base_Data_{run_date_str}.xlsx")

    le.send_email()

print()
print(f" Email sent to gbuehrer@linkedin.com")
print(f"   Subject: CRM Hygiene Run {run_date_str}")
print(f"   Attachments: 4 files")
print()
print("=" * 60)
print(f"  RUN COMPLETE")
print(f"  Run ID:            {RUN_ID}")
print(f"  Run date:          {RUN_DATE}")
print(f"  Rules version:     {RULES_VERSION}")
print(f"  Opps in scope:     {run_log['total_opps_in_scope']}")
print(f"  Flagged opps:      {run_log['distinct_flagged_opps']}")
print(f"  Total rule hits:   {run_log['total_rule_hits']}")
print(f"  Owners impacted:   {run_log['owners_impacted']}")
print(f"  Managers impacted: {run_log['managers_impacted']}")
print(f"  Stagnant opps:     {rule_7_status['rule_7_flagged_opps']}")
print("=" * 60)
print()
print(" Power Automate will:")
print(f"   1. Save files to OneDrive/runs/{run_date_str}/")
print(f"   2. Copy manager_payload to OneDrive/latest/")
print(f"   3. Send emails to {len(manager_payload)} managers")

# ============================================================
# TEAM SUMMARY EMAIL (User-friendly Health Check)
# ============================================================

# 1. Calculate General Metrics
total_opps = run_log['total_opps_in_scope']
flagged_opps = run_log['distinct_flagged_opps']
perc_flagged = round((flagged_opps / total_opps) * 100, 1) if total_opps > 0 else 0

# 2. Calculate Fiscal Year Breakdowns (Safely handling floats/ints)
# Scope
scope_fy26 = len(df[df['close_Year_F'] == 2026])
scope_fy27 = len(df[df['close_Year_F'] == 2027])
scope_fy28_plus = len(df[df['close_Year_F'] >= 2028])

# Flagged
flagged_fy26 = len(flagged_df[flagged_df['close_Year_F'] == 2026])
flagged_fy27 = len(flagged_df[flagged_df['close_Year_F'] == 2027])
flagged_fy28_plus = len(flagged_df[flagged_df['close_Year_F'] >= 2028])

# 3. Extract the Top 3 most frequent rules triggered
top_rules = sorted(run_log['rule_counts'].items(), key=lambda x: x[1], reverse=True)[:3]
top_rules_html = "".join(
    [f"<li style='margin-bottom: 6px;'><b>{rule}:</b> {count} occurrences</li>" 
     for rule, count in top_rules if count > 0]
)
if not top_rules_html:
    top_rules_html = "<li>No hygiene issues found!</li>"

# 4. Extract Flagged Opps by Manager
if not flagged_df.empty:
    manager_counts = flagged_df['Owner_Manager'].value_counts(dropna=False).to_dict()
    manager_html = "".join(
        [f"<li style='margin-bottom: 4px;'><b>{mgr if pd.notna(mgr) else 'Unknown'}:</b> {count} opps</li>" 
         for mgr, count in manager_counts.items()]
    )
else:
    manager_html = "<li>No flagged opportunities to display.</li>"

# 5. Build the HTML body
team_email_body = f"""
<!DOCTYPE html>
<html>
<body style="font-family: Arial, sans-serif; color: #333; line-height: 1.5;">
    <p>Hi Team,</p>
    
    <p><b>Pipeline Pixie</b>🤖 has just finished processing the GAM pipeline. 
    The impacted managers have already received their automated alerts with the opportunities that require action.</p>
    
    <p>Here is our current Health Check:</p>
    
    <h3 style="color: #0073b1; margin-bottom: 8px;">Pipeline Overview</h3>
    <ul style="margin-top: 0;">
        <li style="margin-bottom: 6px;">
            <b>Opportunities in Scope (FY26+ open opps):</b> {total_opps}
            <ul style="margin-top: 2px; font-size: 14px; color: #555;">
                <li>FY26: {scope_fy26} | FY27: {scope_fy27} | FY28+: {scope_fy28_plus}</li>
            </ul>
        </li>
        <li style="margin-bottom: 6px;">
            <b>Flagged Opportunities:</b> {flagged_opps} <i>(~{perc_flagged}% of pipeline)</i>
            <ul style="margin-top: 2px; font-size: 14px; color: #555;">
                <li>FY26: {flagged_fy26} | FY27: {flagged_fy27} | FY28+: {flagged_fy28_plus}</li>
            </ul>
        </li>
        <li style="margin-bottom: 6px;"><b>Managers Notified:</b> {run_log['managers_impacted']}</li>
    </ul>
    
    <h3 style="color: #0073b1; margin-bottom: 8px;">Top 3 Alerts This Week</h3>
    <ul style="margin-top: 0;">
        {top_rules_html}
    </ul>

    <h3 style="color: #0073b1; margin-bottom: 8px;">Flagged Opps by Manager</h3>
    <ul style="margin-top: 0;">
        {manager_html}
    </ul>
    
    <p style="color: #666; font-size: 13px; margin-top: 24px;">
        <i>For a detailed view, the full audit files have been automatically saved to our official SharePoint folder.</i>
    </p>
    
    <p>
        Cheers,<br>
        <b>Gabriel</b>
    </p>
</body>
</html>
"""

# 6. Send the email
with LiEmail() as le_team:
    le_team.sender = "gbuehrer@linkedin.com"
    le_team.recipients = [
        "gbuehrer@linkedin.com"
    ] 
    le_team.subject = f"CRM Hygiene Automation: Weekly Health Check ({run_date_str})"
    le_team.email_body = team_email_body
    le_team.content_type = "html"
    
    le_team.send_email()

print("Team Summary email sent successfully!")

In [72]:
import base64
from IPython.display import HTML, display

def create_download_button(file_path, file_name):
    try:
        with open(file_path, 'rb') as f:
            data = f.read()
        b64 = base64.b64encode(data).decode()
        html = f'''
        <a download="{file_name}" href="data:application/octet-stream;base64,{b64}" 
           target="_blank" 
           style="display:inline-block; padding:10px 20px; background-color:#137cbd; color:white; text-decoration:none; border-radius:4px; margin:5px; font-weight:bold;">
           Download {file_name}
        </a>
        '''
        display(HTML(html))
    except Exception as e:
        print(f" Could not generate link for {file_name}: {e}")

print("Click the buttons below to download the files from this run:")
print("-" * 50)

# Generates the buttons based on the variables already in memory from Block 9
create_download_button(payload_path, f"manager_payload_{run_date_str}.json")
create_download_button(flagged_path, f"flagged_opps_{run_date_str}.xlsx")
create_download_button(run_log_path, f"run_log_{run_date_str}.json")
create_download_button(base_data_path, f"CRM_Hygiene_Base_Data_{run_date_str}.xlsx")
